## Data Preprocessing for (Intent Classification)

In [19]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [20]:
df = pd.read_csv("C:/Invulnerable/pro/realtime-agent-framework/data/customer_support_tickets.csv")


In [21]:
df.shape

(8469, 17)

In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8469 entries, 0 to 8468
Data columns (total 17 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Ticket ID                     8469 non-null   int64  
 1   Customer Name                 8469 non-null   object 
 2   Customer Email                8469 non-null   object 
 3   Customer Age                  8469 non-null   int64  
 4   Customer Gender               8469 non-null   object 
 5   Product Purchased             8469 non-null   object 
 6   Date of Purchase              8469 non-null   object 
 7   Ticket Type                   8469 non-null   object 
 8   Ticket Subject                8469 non-null   object 
 9   Ticket Description            8469 non-null   object 
 10  Ticket Status                 8469 non-null   object 
 11  Resolution                    2769 non-null   object 
 12  Ticket Priority               8469 non-null   object 
 13  Tic

In [23]:
df.head()

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


## Ticket type (Intent Classification)

In [24]:
df_intent = df[["Ticket Subject","Ticket Description","Ticket Type"]].dropna()
print(df_intent.shape)
df_intent.head()

(8469, 3)


,Ticket Subject,Ticket Description,Ticket Type
0,Product setup,I'm having an issue with the {product_purchase...,Technical issue
1,Peripheral compatibility,I'm having an issue with the {product_purchase...,Technical issue
2,Network problem,I'm facing a problem with my {product_purchase...,Technical issue
3,Account access,I'm having an issue with the {product_purchase...,Billing inquiry
4,Data loss,I'm having an issue with the {product_purchase...,Billing inquiry


In [32]:
df_intent["text"] = (
    df_intent["Ticket Subject"].str.lower() + " " +
    df_intent["Ticket Description"].str.lower()
)


In [29]:
df_intent["Ticket Type"].value_counts()

Ticket Type
Refund request          1752
Technical issue         1747
Cancellation request    1695
Product inquiry         1641
Billing inquiry         1634
Name: count, dtype: int64

In [34]:
from sklearn.model_selection import train_test_split

X = df_intent['text']
y = df_intent["Ticket Type"]

X_train,X_test,y_train,y_test = train_test_split(
    X,y,
    test_size=0.2,
    random_state=42,
    stratify = y # it just gives fairness and prevent imbalanced data
)

In [36]:
# NLP Component
# TF-IDF = Term Frequency × Inverse Document Frequency

from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(
    max_features = 20000, # Keeps only the top 20,000 most important terms
    ngram_range=(1,2), # bigram "refund request" this word together will be there in vector
    stop_words="english" # Removes common words like: the, is, and, to, for, with

)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [38]:
from sklearn.linear_model import LogisticRegression

intent_model = LogisticRegression(
    max_iter = 1000,
    n_jobs=-1
)

intent_model.fit(X_train_tfidf,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [39]:
from sklearn.metrics import classification_report , accuracy_score
y_pred = intent_model.predict(X_test_tfidf)

print("Accuracy: ", accuracy_score(y_test,y_pred))
print("\n Classification Report: \n")
print(classification_report(y_test,y_pred))

Accuracy:  0.2166469893742621

 Classification Report: 

                      precision    recall  f1-score   support

     Billing inquiry       0.21      0.21      0.21       327
Cancellation request       0.21      0.18      0.19       339
     Product inquiry       0.24      0.23      0.23       328
      Refund request       0.21      0.23      0.22       351
     Technical issue       0.22      0.24      0.23       349

            accuracy                           0.22      1694
           macro avg       0.22      0.22      0.22      1694
        weighted avg       0.22      0.22      0.22      1694



In [40]:
import numpy as np

feature_names = tfidf.get_feature_names_out()
class_labels = intent_model.classes_

for i, class_label in enumerate(class_labels):
    top_features = np.argsort(intent_model.coef_[i])[-10:]
    print(f"\nTop features for class '{class_label}':")
    print([feature_names[j] for j in top_features])



Top features for class 'Billing inquiry':
['products ve', 'bug hardware', 'new', 'later', 'buy item', 'know', 'good', 'compatibility unable', 'connect', 'apps']

Top features for class 'Cancellation request':
['solution yes', 'file', 'month', 'sent', 'newsletter', 'purchased ve', 'small', 'missing', 'donation', 'thank having']

Top features for class 'Product inquiry':
['99', 'make sure', 'provide', 'buy', 'issue problem', 'trying', 'vendor', 'bug ve', 'options', 'product_p']

Top features for class 'Refund request':
['ordered', 'follow', 'assist experiencing', 'phone', 'help', 'product_name', 'patience', 'solution', 'order', 'gift']

Top features for class 'Technical issue':
['account step', 'thanks issue', 'things', 'feeling', 'assist message', 'assist unable', 'return', 'items', 'assist don', 'provided']


In [47]:
import joblib

joblib.dump(intent_model, "../models/intent_models/ticket_type_model.pkl")
joblib.dump(tfidf,"../models/intent_models/tfidf_vectorizer.pkl")

['../models/intent_models/tfidf_vectorizer.pkl']

## we know that it's accuracy is not so good but we can still use for soft signals 

- predict_proba
- all these probabilities will be combined with metadata based models such as priority prediction to make robust escalation decisions